# San Francisco Housing Production and Family Zoning Plan

## Imports — all in one place

In [ ]:
# Run once from notebooks/ after restarting the kernel.
%cd ..
import sys
sys.path.append(".")

import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from analysis import data_processing, comparisons, visualize


<a id="step-1"></a>
## Step 1 — Clean and filter the data

### 1a. Housing Production

In [ ]:
# Read each permit as text, especially BlockLot (leading zeros are meaningful).
housing = pd.read_csv("data/raw/housing_production.csv", dtype={"bpa": str, "blocklot": str})
housing

In [ ]:
housing_production_filtered = data_processing.clean_housing_data(housing)

In [ ]:
# save filtered housing production dta as csv file

housing_production_filtered.to_csv("data/processed/housing_production_filtered.csv", index=False)

print("Study records:", len(housing_production_filtered))
print("Source net housing additions:", housing_production_filtered["Net Units Completed"].sum())

housing_production_filtered


### 1b. Land Use cleaning from `clean_sf_land_use.py`

In [ ]:
# Read parcel IDs as strings so the Land Use join keeps leading zeros.
land = pd.read_csv("data/raw/San_Francisco_Land_Use.csv",dtype={"mapblklot": str, "residential_units": str}, low_memory=False)
land

In [ ]:
land = data_processing.clean_land_use(land)

In [ ]:
# save filtered land as sf_land_use_clean.csv
land.to_csv("data/processed/sf_land_use_clean.csv", index=False)
print("Cleaned Land Use records:", len(land))

land.head()

<a id="step-2"></a>
## Step 2 — Give each permit a location

### 2a. Exact individual-parcel join — from `tabular_join.py`

Select Land Use rows marked parcel and left-merge Housing Production BlockLot with Land Use mapblklot. Copy the published latitude and longitude, while retaining every housing record. Only coordinates are attached; Land Use stock is never added to permit-level net unit changes.


In [ ]:
# Join Housing Production to single-parcel Land Use centroids by BlockLot.
single = land.loc[land["geography_type"] == "parcel", ["mapblklot", "centroid_latitude", "centroid_longitude"]].rename(columns={"centroid_latitude": "latitude", "centroid_longitude": "longitude"})

located = housing_production_filtered.merge(single, left_on="BlockLot", right_on="mapblklot", how="left")
located = located.drop(columns="mapblklot")
single_match = (~located["latitude"].isna()) & (~located["longitude"].isna())
located["match_status"] = "unmatched"
located["match_note"] = "no_parcel_match"
located.loc[single_match, "match_status"] = "matched_single"
located.loc[single_match, "match_note"] = "parcel_centroid"


### 2b. Multi-parcel records (simple, cautious matching)

Some Land Use rows contain **several quoted BlockLot IDs**, for example `'0123001', '0123002'`. We accept **only complete, quoted lists** with coordinates and **only one distinct candidate location per BlockLot**. Incomplete lists are skipped instead of guessing.

This step can locate a housing record at the **shared group centroid**, not necessarily the exact building. It does not change net housing units.

In [ ]:
multi = land.loc[land["geography_type"] == "multiple_parcels"].copy()
rows = []

for raw, lat, lon in zip(multi["mapblklot"], multi["centroid_latitude"], multi["centroid_longitude"]):
    if pd.isna(lat) or pd.isna(lon):
        continue

    parts = str(raw).split(",")
    parcel_ids = []
    complete_list = True
    for part in parts:
        part = part.strip()
        if len(part) >= 3 and part[0] == "'" and part[-1] == "'":
            parcel_ids.append(part[1:-1])
        else:
            complete_list = False

    if complete_list and len(parcel_ids) >= 2:
        for parcel_id in parcel_ids:
            rows.append([parcel_id, lat, lon])

multi_candidates = pd.DataFrame(rows, columns=["BlockLot", "multi_latitude", "multi_longitude"])
multi_candidates = multi_candidates.drop_duplicates()

# One BlockLot may appear at more than one location: do not guess which is right.
counts = multi_candidates.groupby("BlockLot").size().reset_index(name="locations")
multi_candidates = multi_candidates.merge(counts, on="BlockLot", how="left")
multi_candidates = multi_candidates.loc[multi_candidates["locations"] == 1].copy()

# A left merge keeps every housing record.
located = located.merge(
    multi_candidates[["BlockLot", "multi_latitude", "multi_longitude"]],
    on="BlockLot", how="left"
)
fill = (located["match_status"] == "unmatched") & located["multi_latitude"].notna() & located["multi_longitude"].notna()
located.loc[fill, "latitude"] = located.loc[fill, "multi_latitude"]
located.loc[fill, "longitude"] = located.loc[fill, "multi_longitude"]
located.loc[fill, "match_status"] = "matched_multi_centroid"
located.loc[fill, "match_note"] = "Land Use group centroid"
located = located.drop(columns=["multi_latitude", "multi_longitude"])

print("Multi-parcel matches:", fill.sum())
print("Housing records still unmatched:", (located["match_status"] == "unmatched").sum())

In [ ]:
# Save the initial Land Use matches before supplemental DBI coordinates.
located.to_csv("data/processed/housing_production_located.csv", index=False)

print(located["match_status"].value_counts())
located.head()

### 2c. Simple coordinate recovery

- Keep original Land Use coordinates.
- For still-unmatched records, use a DBI **primary-address coordinate** when its `BPA` has exactly **one unique coordinate**.
- Keep everything else unmatched; never invent a position.

    The raw DBI file is a local JSON input.


In [ ]:
# Remember original Land Use matching results before DBI fills missing points.
located["land_use_match_status"] = located["match_status"]

original_unmatched = located.loc[located["match_status"] == "unmatched"].copy()
print("Before DBI - unmatched records:", len(original_unmatched))
print("Before DBI - unmatched net units:", original_unmatched["Net Units Completed"].sum())

In [ ]:
# Step 2c: Get permit coordinates from DBI

# Read DBI data
with open("data/raw/dbi_all_study_permit_coordinates.json", encoding="utf-8") as file:
    dbi_records = json.load(file)

dbi = data_processing.clean_dbi_coordinates(dbi_records)

In [ ]:
# Step 2c: Fill missing housing coordinates using DBI

# Join DBI coordinates to Housing Production using BPA.
located = located.merge(dbi, on="BPA", how="left")

# Find records with missing coordinates that DBI can provide.
dbi_fill = ((located["latitude"].isna() | located["longitude"].isna()) & located["dbi_latitude"].notna()& located["dbi_longitude"].notna())

# Fill missing housing locations with DBI coordinates.
located.loc[dbi_fill, "latitude"] = located.loc[dbi_fill, "dbi_latitude"]
located.loc[dbi_fill, "longitude"] = located.loc[dbi_fill, "dbi_longitude"]

# Update the matching results.
located.loc[dbi_fill, "match_status"] = "matched_dbi"
located.loc[dbi_fill, "match_note"] = "DBI primary address; one unique coordinate"

# Remove temporary DBI columns.
located = located.drop(columns=["dbi_latitude", "dbi_longitude"])

# Record the final location source based on match_status.
located["location_source"] = located["match_status"].replace({
    "matched_single": "land_use_single",
    "matched_multi_centroid": "land_use_group_centroid",
    "matched_dbi": "dbi_primary",
    "unmatched": "unlocated"
})

# Check the final matching results.
print("Records recovered using DBI:", dbi_fill.sum())
print("Total housing records:", len(located))
print(located["location_source"].value_counts())

print("Net units still unmatched:", located.loc[located["match_status"] == "unmatched", "Net Units Completed"].sum())


In [ ]:
# Preserve unmatched records for the professor's validation checks.
unmatched = located.loc[located["match_status"] == "unmatched"].copy()
unmatched.to_csv("data/processed/unmatched_housing_records.csv", index=False)
located.to_csv("data/processed/housing_production_located_recovered.csv", index=False)

location_summary = located.groupby("location_source")["Net Units Completed"].agg(["count", "sum"])
location_summary.to_csv("data/processed/location_source_summary.csv")
location_summary

<a id="step-3"></a>
## Step 3 — Assign located housing records to census tracts

Use `Point` and `gpd.sjoin()` to place each housing record inside a 2020 Census Tract. **The spatial join is based on coordinates; unmatched rows remain in `detailed` but have no GEOID.**

Use the source's unique `source_row_id` to merge the GEOID back (not `BPA`, which might be repeated). Report the fraction of records and **net housing units** that remain unassigned. This is essential because we simplified Step 2c; do **not** describe the output as a complete citywide map when records lack a tract.

In [ ]:
# Step 3: Assign Housing Production Records to 2020 Census Tracts(spatial join)

# Import Point to create geographic points from coordinates.
from shapely.geometry import Point

# Load the 2020 Census Tract 
tracts = gpd.read_file("data/raw/census_2020_tracts_raw.geojson")

# Rename the columns to match the names used in our project.
tracts = tracts.rename(columns={"geoid": "GEOID", "name": "NAME"})

# Convert tract geometries to EPSG:4326 (longitude and latitude).
tracts = tracts.to_crs("EPSG:4326")

# Give each housing record a unique ID for the tract join.
located["source_row_id"] = range(len(located))
has_coordinates = located["latitude"].notna() & located["longitude"].notna()

# Keep the source_row_id of records with valid coordinates.
# source_row_id identifies each original Housing Production row.
point_rows = located.loc[located["latitude"].notna() & located["longitude"].notna(), ["source_row_id"]].copy()

# Create an empty list to store geographic Point objects.
points = []

# Loop through the longitude and latitude of each located housing record.
# zip() pairs each longitude with its corresponding latitude.
for lon, lat in zip(located.loc[has_coordinates, "longitude"], located.loc[has_coordinates, "latitude"]):

    # Create a geographic point and add it to the list.
    # Point() takes longitude first, then latitude.
    points.append(Point(lon, lat))

# Create a GeoDataFrame containing the housing points.
# Each row has a source_row_id and a Point geometry.
housing_points = gpd.GeoDataFrame(
    point_rows,
    geometry=points,
    crs="EPSG:4326"
)

# Spatial Join: assign each housing point to a Census Tract.
# "within" checks whether the point falls inside a tract polygon.
# how="left" keeps all housing points, even if no tract is matched.
housing_joined = gpd.sjoin(
    housing_points,
    tracts[["GEOID", "geometry"]],
    how="left",
    predicate="within"
)

# Count the number of housing records with coordinates.
print("Housing points:", len(housing_points))

# Check the number of rows after the Spatial Join.
# This helps identify whether any housing points matched multiple tracts.
print("Rows after spatial join:", len(housing_joined))

# Check whether source_row_id is duplicated in the original located DataFrame.
# Each source_row_id should identify one Housing Production record.
print("Repeated source IDs after spatial join:", housing_joined["source_row_id"].duplicated().sum())

# Join the Census Tract GEOID back to the original Housing Production records.
# Match using source_row_id, keeps all original records, including those without a GEOID.
detailed = located.merge(housing_joined[["source_row_id", "GEOID"]], on="source_row_id",how="left")

# Save the Housing Production records with their assigned Census Tract GEOIDs.
detailed.to_csv("data/processed/housing_production_with_tract.csv",index=False)

# Compare Land Use matching and final DBI + Spatial Join
# Identify matched records at each stage
land_match = detailed["land_use_match_status"] != "unmatched"
final_match = detailed["GEOID"].notna()

# Housing units
units = detailed["Net Units Completed"]

# Create a summary table
join_summary = pd.DataFrame({
    "stage": ["Land Use only", "After DBI + tract join"],
    "assigned_records": [land_match.sum(), final_match.sum()],
    "unassigned_records": [(~land_match).sum(), (~final_match).sum()],
    "assigned_net_units": [units[land_match].sum(), units[final_match].sum()],
    "unassigned_net_units": [units[~land_match].sum(), units[~final_match].sum()]
})

# Save and display results
join_summary.to_csv("data/processed/housing_join_summary.csv", index=False)
join_summary


<a id="step-4"></a>
## Step 4 — Build a tract-level table

Use all 2020 Census Tracts and add up **signed** housing changes from records that received a `GEOID`. Keep housing additions and losses separately. Unassigned records stay in `detailed` and in the validation summary; **do not silently treat them as zero**. A current Land Use snapshot is not a 2005 housing-stock baseline.

In [ ]:
# Step 4: Aggregate housing production by Census Tract

# Keep only housing records that have been assigned a Census Tract GEOID.
# .copy() creates a separate DataFrame without changing detailed.
assigned = detailed.loc[~detailed["GEOID"].isna()].copy()

# Create two new columns to separately track housing additions and losses.
# Start both columns at zero for every housing record.
assigned["positive_net_units"] = 0
assigned["negative_net_units_abs"] = 0

# For records with positive Net Units Completed, record the units added.
assigned.loc[assigned["Net Units Completed"] > 0,"positive_net_units"] = assigned["Net Units Completed"]

# For records with negative Net Units Completed, record the housing losses.
# Multiply by -1 so losses are stored as positive numbers.
assigned.loc[assigned["Net Units Completed"] < 0,"negative_net_units_abs"] = -assigned["Net Units Completed"]

# Group housing records by GEOID and sum three columns:
# 1. Total positive housing additions
# 2. Total housing losses (as positive numbers)
# 3. Signed net housing additions (additions minus losses)
tract_sums = assigned.groupby("GEOID")[["positive_net_units", "negative_net_units_abs", "Net Units Completed"]].sum().reset_index()

# Rename the net housing column to describe the 2005-2025 study period.
tract_sums = tract_sums.rename(columns={"Net Units Completed": "net_units_added_2005_2025"})

# Count how many Housing Production records belong to each Census Tract.
tract_records = assigned.groupby("GEOID").size().to_frame(name="production_records")

# Join the record counts to the housing unit totals using GEOID.
tract_sums = tract_sums.merge(tract_records, on="GEOID", how="left")

# Join housing production totals to the Census Tract GeoDataFrame.
# Use a left join to keep all Census Tracts, including those with no
# assigned housing production records.
tract_production = tracts.merge(tract_sums, on="GEOID", how="left")

# List the four columns that contain housing counts and record counts.
count_columns = [
    "positive_net_units",
    "negative_net_units_abs",
    "net_units_added_2005_2025",
    "production_records"
]

# Replace missing values with 0 for tracts with no matched records.
# Convert these columns to integers because they represent unit counts.
tract_production[count_columns] = (tract_production[count_columns].fillna(0).astype(int))

# Rename NAME to tract_name.
tract_production = tract_production.rename(columns={"NAME": "tract_name"})


<a id="step-5"></a>
## Step 5 — Calculate 2005–2025 signed net housing additions

Save and map signed net housing additions for 2005–2025. Net additions equal positive net changes minus the absolute value of negative net changes. Use the full selected housing sample as the citywide total, keeping unassigned housing separate. These are completed net housing changes, not permit counts, parcel stock, or percentage growth rates. No unverified historical-stock denominator is used.


In [ ]:
# Keep full-source and unassigned units separate; the tract table excludes unlocated records.
tract_production.drop(columns="geometry").to_csv("data/processed/tract_housing_production.csv", index=False)
city_net_units = detailed["Net Units Completed"].sum()
unassigned_net_units = detailed.loc[detailed["GEOID"].isna(), "Net Units Completed"].sum()


print("Full-source net additions:", city_net_units)
print("Net additions assigned to tracts:", tract_production["net_units_added_2005_2025"].sum())
print("Net additions without a tract:", unassigned_net_units)


In [ ]:
visualize.plot_housing_map(tract_production)

<a id="step-6"></a>
## Step 6 — Classify tracts using ACS 2020–2024 

clean B25003, clean B19013, merge by GEOID, then use `pd.qcut()` to divide tracts into three income groups and three renter-share groups. We use the **same ACS 2024 five-year data** already in this project's `data/raw` folder, so no extra copies or folder changes are needed. The groups describe **2020–2024 conditions**, not neighborhood conditions in 2005.

**Limitation:** Income and renter-share groups reflect 2020–2024 ACS conditions. Neighborhood characteristics may have changed during 2005–2025. The comparisons describe historical housing production grouped by recent neighborhood conditions; they do not establish the income or renter share at the time housing was completed.


In [ ]:
tenure = pd.read_csv("B25003/ACSDT5Y2024.B25003-Data.csv", skiprows=[1])
tenure

In [ ]:
# Step 6a: Read 2020-2024 ACS tenure (B25003).
tenure = pd.read_csv("B25003/ACSDT5Y2024.B25003-Data.csv", skiprows=[1])
tenure = data_processing.clean_tenure_data(tenure)

# Renter share = renter-occupied units / total occupied units.
tenure["renter_share"] = tenure["renter"] / tenure["total"]

tenure = tenure[["GEOID", "renter_share"]]

print("Valid tenure tracts:", len(tenure))

In [ ]:
# Step 6b: Read 2020-2024 ACS median income (B19013).
income = pd.read_csv("data/raw/ACSDT5Y2024.B19013-Data.csv", skiprows=[1])
income

In [ ]:
income_clean = data_processing.clean_income_data(income)
income_clean.head()

In [ ]:
# Step 6c: Merge the two ACS tables by tract GEOID.
merged = tenure.merge(income_clean, on="GEOID", how="inner")

# Divide the same tracts into three equal-sized income and renter-share groups.
merged["income_group"] = pd.qcut(merged["median_income"], 3, labels=["low", "middle", "high"])
merged["renter_group"] = pd.qcut(merged["renter_share"], 3, labels=["low_renter", "middle_renter", "high_renter"])

print("Tracts with both ACS measures:", len(merged))
print("Income groups:\n", merged["income_group"].value_counts().sort_index())
print("Renter groups:\n", merged["renter_group"].value_counts().sort_index())


In [ ]:
# Join ACS groups with Housing Production data

# Merge ACS income and renter groups with housing production by GEOID.
# Keep all Census Tracts using a left join.
master = tract_production.merge(merged[["GEOID", "median_income", "renter_share", "income_group", "renter_group"]], on="GEOID",how="left")

# Replace missing ACS groups with "unclassified".
for column in ["income_group", "renter_group"]:
    groups = []
    for group in master[column]:
        if pd.isna(group):
            groups.append("unclassified")
        else:
            groups.append(group)

    master[column] = groups

# Save the results without the geometry column.
master.drop(columns="geometry").to_csv("data/processed/tract_acs_groups.csv", index=False)

# Check the number of Census Tracts.
print("Total tracts:", len(master))
print("Tracts with ACS groups:", len(merged))

# Display the first five rows.
master[["GEOID", "median_income", "renter_share", "income_group", "renter_group"]].head()


<a id="step-7"></a>
## Step 7 — Housing additions by 2020–2024 ACS groups

Sum the **2005–2025 signed net additions** from Steps 4–5 for the income and renter-share groups from Step 6.
Keep unclassified tracts and unassigned housing records visible in the summary. The ACS groups describe **2020–2024**, not 2005.


In [ ]:
# Step 7: Reuse the same group summary for both classifications.
income_summary = comparisons.summarize_group(master, "income_group", "income")
renter_summary = comparisons.summarize_group(master, "renter_group", "renter")

# Housing records without GEOID are outside both group comparisons.
unassigned = pd.DataFrame({
    "group": ["unassigned_to_tract", "unassigned_to_tract"],
    "tracts": [0, 0],
    "production_records": [detailed["GEOID"].isna().sum()] * 2,
    "net_units": [unassigned_net_units] * 2,
    "dimension": ["income", "renter"]
})

group_comparison = pd.concat([income_summary, renter_summary, unassigned], ignore_index=True)
group_comparison["share_city_net_pct"] = group_comparison["net_units"] / city_net_units * 100

group_comparison.to_csv("data/processed/group_housing_comparisons.csv", index=False)
group_comparison

In [ ]:
visualize.plot_group_comparison(group_comparison, unassigned_net_units)

## Extra visualization A — Distribution of net housing additions (histogram + KDE)

This uses the **same histogram/KDE style** as the sample `baseline_2005_stock` plot, but it uses the housing variable actually available in this project: `net_units_added_2005_2025`. It must **not** be labeled "2005 baseline housing stock" because the project does not have a verified 2005 stock estimate.

In [ ]:
visualize.plot_net_additions_distribution(tract_production)

## Extra visualization B — Top 10 tracts by net housing additions (horizontal bar chart)

Same layout as the sample `growth_rate_pct` plot; because no reliable `baseline_2005_stock` exists, the chart ranks **absolute net housing additions**, not percentage housing growth.

In [ ]:
visualize.plot_top_tracts(tract_production)